In [ ]:
from utils.supabase_utils import extract_all_rows
from utils.feature_transform import get_features
from config import (
    FEEDBACK_NAME,
    CATEGORY_NAME,
    FEATURE_REGISTRY_NAME,
    FEATURE_REGISTRY_ID
)


In [2]:
category_df = extract_all_rows(CATEGORY_NAME)
category_df

,category_id,category
0,8bfa06c2-f7fb-45da-a2a9-d9031e5f403b,lunch
1,9f82d159-3127-45db-8e01-0b50b44e15f3,apply job
2,4d2b380a-3c10-47f5-98f9-dba58edb18bd,exercise
3,c59c0d90-d940-4f0c-a29a-c48c0bf6d9f9,dinner/drinks
4,80c6d56c-a1b6-4151-b84a-5d1cdcff5f9f,breakfast
5,03f807d8-cb08-446f-ab7b-ab6276d97442,work/career fair
6,0643b054-ff74-4e7e-8b9d-ca5fbca81ac3,others


In [3]:
feedback_df = extract_all_rows(FEEDBACK_NAME)
feedback_df

,feedback_id,created_at,pred_min,meeting_location,date,meeting_time,arrived_time,meeting_lat,meeting_lon,category_id,model_id
0,8b501853-1739-479e-9094-55c2ad7e2396,2026-05-25T14:25:42.447675+00:00,21,"Pasir Panjang, 119, Pasir Panjang Road, Pasir ...",2026-05-29,2026-05-29T20:30:00+00:00,2026-05-25T23:00:00+00:00,1.276201,103.791476,c59c0d90-d940-4f0c-a29a-c48c0bf6d9f9,93b1785d-ee10-4669-9958-caaf63832af6
1,f6927a1a-ea36-4107-980e-18e1348389e1,2026-05-27T03:23:31.080553+00:00,19,Bukit Panjang Plaza,2026-05-06,2026-05-06T15:30:00+00:00,2026-05-06T18:30:00+00:00,1.350000,103.900000,c59c0d90-d940-4f0c-a29a-c48c0bf6d9f9,93b1785d-ee10-4669-9958-caaf63832af6
2,f3c9b8d2-784a-470d-9f17-cfc46b816593,2026-05-27T03:38:34.6268+00:00,48,"Choa Chu Kang, West Region, Singapore",2026-05-27,2026-05-27T18:30:00+00:00,2026-05-27T18:45:00+00:00,1.385719,103.744588,4d2b380a-3c10-47f5-98f9-dba58edb18bd,93b1785d-ee10-4669-9958-caaf63832af6


In [8]:
modified_feedback_df = get_features(feedback_df)

modified_feedback_df = (
    modified_feedback_df
    .merge(
        category_df,
        how="left",
        on="category_id"
    )
    .drop(columns=["category_id"])
)

modified_feedback_df

,feedback_id,day_of_week,time_of_day,distance_km,act_min,pred_min,category
0,f6927a1a-ea36-4107-980e-18e1348389e1,2,evening,16.578377,180.0,19,dinner/drinks
1,f3c9b8d2-784a-470d-9f17-cfc46b816593,2,evening,1.153625,15.0,48,exercise
2,8b501853-1739-479e-9094-55c2ad7e2396,4,morning,12.514021,-5610.0,21,dinner/drinks


In [9]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

modified_feedback_df["error"] = modified_feedback_df["act_min"] - modified_feedback_df["pred_min"]

modified_feedback_df

,feedback_id,day_of_week,time_of_day,distance_km,act_min,pred_min,category,error
0,f6927a1a-ea36-4107-980e-18e1348389e1,2,evening,16.578377,180.0,19,dinner/drinks,161.0
1,f3c9b8d2-784a-470d-9f17-cfc46b816593,2,evening,1.153625,15.0,48,exercise,-33.0
2,8b501853-1739-479e-9094-55c2ad7e2396,4,morning,12.514021,-5610.0,21,dinner/drinks,-5631.0
